# 00 · v1 at a glance

**What you will see:** the twelve parts of v1 and the gate each one passes; the package measured against v0.5; the ten
names a program imports; and what still needs a real arm, credentials or a GPU.

In [1]:
import re, subprocess, sys, os, tomllib
from pathlib import Path
import _nb
import strands_robots

ROOT = _nb.ROOT
V05 = Path(os.environ.get("STRANDS_ROBOTS_V05", Path.home() / "robots"))  # a v0.5 checkout, optional
print("v1  :", strands_robots.__version__, "from", Path(strands_robots.__file__).parent.relative_to(ROOT))
print("v0.5:", subprocess.run(["git", "-C", V05, "log", "-1", "--format=%h %cs"], capture_output=True, text=True).stdout.strip() if V05.exists() else "(no checkout; set STRANDS_ROBOTS_V05)")
_nb.show(_nb.drawing("v08_parts"))

v1  : 1.0.0.dev0 from strands_robots
v0.5: 0e6cf5055 2026-10-06


## The parts

Each part is one pull request on `dev` with its own gate. The table is the one on the docs site's Status page
(`docs/status.md`), read straight from it.

In [2]:
status = (ROOT / "docs" / "status.md").read_text()
rows = []
for line in status.splitlines():
    cells = [c.strip() for c in line.strip().strip("|").split("|")]
    if len(cells) == 3 and re.match(r"^\d", cells[0]):
        rows.append(dict(zip(("part", "what it adds", "gate"), cells)))
_nb.show(_nb.table(rows))
print("whole suite:", re.search(r"`pytest -q tests` → \*\*(.+?)\*\*", status).group(1))

| part | what it adds | gate |
|---|---|---|
| 0 | an empty package on `dev` | — |
| 1 | types + `Description` | — |
| 2 | `Simulation` + MuJoCo | acceptance 1, 2 |
| 3 | `Robot` is one class for sim and hardware | — |
| 4 | `Policy`: any LeRobot checkpoint, in-process or over Zenoh | acceptance 3, 4 |
| 4b | ONNX skills (`Policy("holosoma")`, the microduck skills) | G1 stands 10 s, the duck walks 1.29 m; acceptance 5 |
| 5 | drivers, the free twin, the operator gate | acceptance 10; 9 needs a real SO-101 |
| 5.5 | a Robot is a device: manifest, safety chain, estop | acceptance 11, 13 |
| 6 | Isaac Sim and Newton backends | 7/7 on an L40S (skipped without CUDA) |
| 7 | the mesh: `Mesh`, `use_mesh`, Zenoh and AWS IoT | acceptance 7 |
| 8 | agent tools generated from the manifest | acceptance 12, 14 |
| 9 | `run(record=)` writes LeRobot v3 datasets | acceptance 6, 15 |
| 10 | the dashboard: the world in a browser tab | acceptance 8 |
| 11 | `dev` becomes `main` | — |
| 12 | these docs and the notebooks | `mkdocs build --strict`, every page's first example runs |

whole suite: 566 passed, 25 skipped


A few of those checks, re-run right now (the package imports nothing heavy, the public surface, every file ≤ 250 lines):

In [3]:
env = {**os.environ, "PYTHONPATH": str(ROOT), "PYTHONDONTWRITEBYTECODE": "1"}
out = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no",
     "tests/test_package.py", "tests/test_public_surface.py", "tests/test_layout.py"],
    cwd=ROOT, env=env, capture_output=True, text=True,
)
print(out.stdout.strip().splitlines()[-1])
assert out.returncode == 0

8 passed in 1.04s


## The package, measured

One concept is one package. Lines are every line of every `.py` file, counted the same way in both trees.

In [4]:
def lines(p):
    return sum(len(f.read_text(errors="ignore").splitlines()) for f in p.rglob("*.py") if "__pycache__" not in f.parts)

def files(p):
    return sum(1 for f in p.rglob("*.py") if "__pycache__" not in f.parts)

pkg = ROOT / "strands_robots"
v1 = {d.name: (files(d), lines(d)) for d in sorted(pkg.iterdir()) if d.is_dir() and d.name != "__pycache__" and files(d)}
top = sum(len(f.read_text().splitlines()) for f in pkg.glob("*.py"))
_nb.show(_nb.table([{"v1 package": f"`{k}/`", "files": n, "lines": f"{l:,}"} for k, (n, l) in sorted(v1.items(), key=lambda kv: -kv[1][1])]
                   + [{"v1 package": "top-level modules", "files": len(list(pkg.glob('*.py'))), "lines": top}]))
v1_total = lines(pkg)
biggest = max((len(f.read_text().splitlines()), f.relative_to(ROOT)) for f in pkg.rglob("*.py"))
print(f"v1 total: {v1_total:,} lines in {files(pkg)} files; largest file {biggest[1]} = {biggest[0]} lines")

| v1 package | files | lines |
|---|---|---|
| `robot/` | 30 | 2,515 |
| `mesh/` | 20 | 2,226 |
| `simulation/` | 21 | 1,850 |
| `policy/` | 17 | 1,204 |
| `dashboard/` | 8 | 859 |
| `description/` | 9 | 554 |
| `tools/` | 7 | 426 |
| `recording/` | 5 | 273 |
| `units/` | 5 | 263 |
| `types/` | 4 | 160 |
| top-level modules | 4 | 129 |

v1 total: 10,459 lines in 130 files; largest file strands_robots/mesh/mesh.py = 247 lines


In [5]:
if (V05 / "strands_robots").exists():
    old = V05 / "strands_robots"
    v05 = sorted(((lines(d), d.name) for d in old.iterdir() if d.is_dir() and d.name != "__pycache__"), reverse=True)
    shown, rest = v05[:8], v05[8:]
    old_top = sum(len(f.read_text(errors="ignore").splitlines()) for f in old.glob("*.py"))
    rows = [{"v0.5 package": f"`{n}/`", "lines": f"{l:,}"} for l, n in shown]
    rows += [{"v0.5 package": f"{len(rest)} more packages", "lines": f"{sum(l for l, _ in rest):,}"},
             {"v0.5 package": f"{len(list(old.glob('*.py')))} top-level modules", "lines": f"{old_top:,}"}]
    _nb.show(_nb.table(rows))
    v05_total = lines(old)
    print(f"v0.5 total: {v05_total:,} lines  ->  v1: {v1_total:,} lines ({v05_total / v1_total:.0f}x smaller)")
else:
    print(f"no v0.5 checkout here; measured on 2026-10-15: v0.5 258,503 lines -> v1 {v1_total:,} lines")

| v0.5 package | lines |
|---|---|
| `simulation/` | 76,433 |
| `drivers/` | 34,540 |
| `policies/` | 31,248 |
| `dashboard/` | 23,618 |
| `mesh/` | 21,629 |
| `tools/` | 17,351 |
| `training/` | 13,734 |
| `rendering/` | 3,587 |
| 7 more packages | 12,134 |
| 38 top-level modules | 24,229 |

v0.5 total: 258,503 lines  ->  v1: 10,459 lines (25x smaller)


How the packages may import each other is a written contract, checked on every change (`lint-imports`):

In [6]:
_nb.show(_nb.drawing("d05_packages"))
contracts = tomllib.loads((ROOT / "pyproject.toml").read_text())["tool"]["importlinter"]["contracts"]
for c in contracts:
    print("·", c["name"])

· types is a leaf: it imports nothing else in the package
· description depends only on types, _gl, _extras
· simulation does not import robot or policy (except the lazy Simulation.run edge)
· policy imports neither simulation nor the drivers
· the mesh never imports policy
· robot.safety and robot.device import no policy, no simulation backend
· recording depends only on types, description and units (the run loop calls it, never the reverse)
· units depends only on types and description


## Ten names

A program imports these from the top level; everything else comes from its package
(`from strands_robots.simulation import Scene`). Importing the package loads no physics, no torch, no zenoh.

In [7]:
import strands_robots as sr
heavy = ("mujoco", "torch", "lerobot", "zenoh", "onnxruntime", "isaacsim", "warp")
print(len(sr.__all__), "names:", ", ".join(sr.__all__))
print("heavy modules loaded by `import strands_robots`:", [m for m in heavy if m in sys.modules] or "none")
def kind(n):
    v = getattr(sr, n)
    return "str" if n == "__version__" else "class" if isinstance(v, type) else "type alias"
def says(n):
    v = getattr(sr, n)
    if n == "__version__":
        return v
    if not isinstance(v, type):  # Action is a union, not a class
        return "a vector in `action_keys` order, or a `{joint: target}` mapping: " + str(v).replace("|", "or")
    return (v.__doc__ or "").strip().splitlines()[0]
_nb.show(_nb.table([
    {"name": n, "is": kind(n),
     "from": "strands_robots.types" if kind(n) == "type alias" else getattr(getattr(sr, n), "__module__", "strands_robots"),
     "says": says(n)}
    for n in sr.__all__]))

10 names: Robot, Simulation, Policy, Body, Description, Observation, Action, JointState, Pose, __version__
heavy modules loaded by `import strands_robots`: none


| name | is | from | says |
|---|---|---|---|
| Robot | class | strands_robots.robot.robot | A body in a world (``sim=``, or ``Simulation.default()``) or on a bus (``port=``/``driver=``): same verbs. |
| Simulation | class | strands_robots.simulation.simulation | One world. Owns time. Holds many robots. |
| Policy | class | strands_robots.policy.base | Base class: override ``act``; ``units`` and ``requirements`` say what the policy speaks and needs. |
| Body | class | strands_robots.robot.body | Something with joints you can read and targets you can write. |
| Description | class | strands_robots.description.description | A robot model resolved to a file plus the facts a world or a bus needs about it. |
| Observation | class | strands_robots.types.observation | What a body reports at time ``t``: joints, base pose, camera images and per-body extras. |
| Action | type alias | strands_robots.types | a vector in `action_keys` order, or a `{joint: target}` mapping: numpy.ndarray or collections.abc.Mapping[str, float] or strands_robots.types.joints.JointState |
| JointState | class | strands_robots.types.joints | Positions/velocities/efforts for a named set of joints (SI: rad, rad/s, N·m). |
| Pose | class | strands_robots.types.pose | A rigid pose in the world frame: position (m) and orientation as a unit quaternion (w, x, y, z). |
| __version__ | str | strands_robots | 1.0.0.dev0 |

In [8]:
import ast
if (V05 / "strands_robots").exists():
    tree = ast.parse((V05 / "strands_robots" / "__init__.py").read_text())
    n05 = next(len(a.value.elts) for a in ast.walk(tree) if isinstance(a, ast.Assign) and any(getattr(t, "id", "") == "__all__" for t in a.targets))
else:
    n05 = 50  # measured on 2026-10-15 against v0.5 main
print(f"v0.5 exported {n05} top-level names; v1 exports {len(sr.__all__)}")

v0.5 exported 50 top-level names; v1 exports 10


## What still needs a real arm, credentials or a GPU

Everything above runs on a laptop. These rows do not:

In [9]:
_nb.show(_nb.table([
    {"needs": "real SO-101 follower", "what": "read joints from the bus (acceptance check 9)",
     "how": "`STRANDS_SO101_PORT=… STRANDS_SO101_CALIBRATION=… pytest tests/acceptance/test_09*`"},
    {"needs": "real SO-101 follower", "what": "estop on the physical arm; broadcast estop stops the last goal write; "
     "controller killed mid-motion → `on_disconnect`", "how": "sim analogues green (estop acked p50 1.15 ms, watchdog 3-4 ms)"},
    {"needs": "real SO-101 follower", "what": "an agent `act` behind the operator gate on hardware",
     "how": "green on a fake serial bus; needs the arm"},
    {"needs": "Hub token", "what": "`push=True` upload of a recorded dataset", "how": "parsed + stubbed, nothing ever uploaded"},
    {"needs": "Hub gate", "what": "FLUX.3 SO-101 checkpoint in the policy matrix", "how": "accept the gate on the Hub (403 today)"},
    {"needs": "AWS creds", "what": "real AWS IoT round trip, real Bedrock agent turn",
     "how": "done with fresh creds: IoT discover 0.83 s, act p50 203 ms; Bedrock turn moved shoulder_pan to 0.47"},
    {"needs": "CUDA GPU", "what": "`Simulation(\"isaac\")`, `physics=\"newton\"` (16 tests skip on the Mac)",
     "how": "7/7 green on an L40S; Newton 1,754,372 env-steps/s at 1024 envs"},
    {"needs": "CUDA GPU", "what": "every LeRobot family over Zenoh", "how": "L40S: 13 of 25 families admitted, the rest refused verbatim"},
    {"needs": "CUDA GPU + setup", "what": "the tiny ACT checkpoint on Isaac", "how": "not run (no lerobot in the Isaac Python)"},
]))

| needs | what | how |
|---|---|---|
| real SO-101 follower | read joints from the bus (acceptance check 9) | `STRANDS_SO101_PORT=… STRANDS_SO101_CALIBRATION=… pytest tests/acceptance/test_09*` |
| real SO-101 follower | estop on the physical arm; broadcast estop stops the last goal write; controller killed mid-motion → `on_disconnect` | sim analogues green (estop acked p50 1.15 ms, watchdog 3-4 ms) |
| real SO-101 follower | an agent `act` behind the operator gate on hardware | green on a fake serial bus; needs the arm |
| Hub token | `push=True` upload of a recorded dataset | parsed + stubbed, nothing ever uploaded |
| Hub gate | FLUX.3 SO-101 checkpoint in the policy matrix | accept the gate on the Hub (403 today) |
| AWS creds | real AWS IoT round trip, real Bedrock agent turn | done with fresh creds: IoT discover 0.83 s, act p50 203 ms; Bedrock turn moved shoulder_pan to 0.47 |
| CUDA GPU | `Simulation("isaac")`, `physics="newton"` (16 tests skip on the Mac) | 7/7 green on an L40S; Newton 1,754,372 env-steps/s at 1024 envs |
| CUDA GPU | every LeRobot family over Zenoh | L40S: 13 of 25 families admitted, the rest refused verbatim |
| CUDA GPU + setup | the tiny ACT checkpoint on Isaac | not run (no lerobot in the Isaac Python) |

## Where this sits in v1

This is the map. Notebooks 01-08 walk the same SO-101 arm (with its G1 neighbour) through each part, in order.